# Telenet Mailbox Scanner — Subscriptions & Contacts

This notebook connects to your **stijn.huysman@telenet.be** mailbox over IMAP and:

1. Lists your mail folders (so you can confirm the "Sent" folder name)
2. Scans message **headers only** (fast — no bodies downloaded) in Inbox + Sent
3. Detects likely **subscriptions / online services** using the `List-Unsubscribe` header and common sender patterns
4. Builds a **contact list** of everyone who emailed you and everyone you emailed — for your "I have a new email address" heads-up
5. Exports two Excel files: `subscriptions.xlsx` and `contacts.xlsx`

## Before you run it
- Run this on your own PC (you selected local Python/Jupyter) — your password is typed locally via a hidden prompt and only goes to Telenet's IMAP server over SSL. Nothing is sent anywhere else.
- If Telenet ever asks you to generate a dedicated **app password** for mail clients, use that instead of your normal webmail password.
- A "big mailbox" can mean thousands of messages — the header-only fetch keeps this reasonably fast, but a full scan of tens of thousands of messages can still take several minutes. There's a `MAX_MESSAGES` limit below you can raise once you've tested it on a small batch.


In [1]:
import imaplib
import email
from email.header import decode_header
from email.utils import getaddresses, parsedate_to_datetime
import getpass
import re
import pandas as pd
from collections import defaultdict

IMAP_SERVER = "imap.telenet.be"
IMAP_PORT = 993
EMAIL_ADDRESS = "stijn.huysman@telenet.be"

# Safety cap while testing. Set to None to scan everything once you've validated it works.
MAX_MESSAGES = 500


In [3]:
EMAIL_PASSWORD = getpass.getpass("Telenet email password (hidden input): ")

imap = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
imap.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
print("Connected as", EMAIL_ADDRESS)


Connected as stijn.huysman@telenet.be


## Step 1 — List folders

Run this first so you can see the *exact* folder name for your Sent items — Dutch Telenet webmail often calls it something like `Verzonden Items` rather than `Sent`. Copy the correct name into `SENT_FOLDER_NAME` in the next cell.

In [4]:
status, folders = imap.list()
for f in folders:
    print(f.decode(errors="replace"))


(\HasChildren) "/" "0 geclasseerd"
(\HasNoChildren) "/" "0 geclasseerd/MASTAT"
(\HasNoChildren) "/" "0 geclasseerd/smartschool"
(\HasNoChildren) "/" "Archief"
(\HasNoChildren) "/" "Archive"
(\HasNoChildren \Drafts) "/" "Drafts"
(\HasNoChildren) "/" "Gutami"
(\HasChildren) "/" "INBOX"
(\HasNoChildren) "/" "INBOX/0 VERKOOP"
(\HasChildren) "/" "INBOX/0. Antoon"
(\HasNoChildren) "/" "INBOX/0. Antoon/medeleven"
(\HasNoChildren) "/" "INBOX/0. Antoon/tiptop"
(\HasNoChildren) "/" "INBOX/0. Antoon/total"
(\HasNoChildren) "/" "INBOX/0. Appostolinen"
(\HasNoChildren) "/" "INBOX/0. arc"
(\HasChildren) "/" "INBOX/0. Basket"
(\HasNoChildren) "/" "INBOX/0. Basket/refs"
(\HasNoChildren) "/" "INBOX/0. CECILE"
(\HasNoChildren) "/" "INBOX/00. Koppenberg"
(\HasNoChildren) "/" "INBOX/00. LIMA"
(\HasNoChildren) "/" "INBOX/00. mercedes"
(\HasNoChildren) "/" "INBOX/00.reis"
(\HasChildren) "/" "INBOX/1. Coupure verhuur"
(\HasNoChildren) "/" "INBOX/1. Coupure verhuur/1a Coupure"
(\HasChildren) "/" "INBOX/5 pers

In [5]:
# Update these two based on the folder listing above
INBOX_FOLDER_NAME = "INBOX"
SENT_FOLDER_NAME = "Sent"   # e.g. could be "Verzonden Items", "INBOX.Sent", "Sent Items"


## Step 2 — Header-fetching helper

In [6]:
def decode_str(value):
    """Decode a possibly-encoded email header into plain text."""
    if not value:
        return ""
    parts = decode_header(value)
    out = []
    for text, enc in parts:
        if isinstance(text, bytes):
            try:
                out.append(text.decode(enc or "utf-8", errors="replace"))
            except LookupError:
                out.append(text.decode("utf-8", errors="replace"))
        else:
            out.append(text)
    return "".join(out)


def fetch_headers(imap_conn, folder_name, max_messages=None):
    """Fetch From/To/Cc/Subject/Date/List-Unsubscribe headers for all messages in a folder."""
    status, _ = imap_conn.select(f'"{folder_name}"', readonly=True)
    if status != "OK":
        print(f"Could not open folder: {folder_name}")
        return []

    status, data = imap_conn.search(None, "ALL")
    if status != "OK":
        print(f"Search failed on folder: {folder_name}")
        return []

    ids = data[0].split()
    if max_messages:
        ids = ids[-max_messages:]  # most recent N

    print(f"{folder_name}: {len(ids)} messages to scan")

    records = []
    header_fields = "(FROM TO CC SUBJECT DATE LIST-UNSUBSCRIBE)"
    batch_size = 200

    for i in range(0, len(ids), batch_size):
        batch = ids[i:i + batch_size]
        id_set = b",".join(batch)
        status, msg_data = imap_conn.fetch(id_set, f"(BODY.PEEK[HEADER.FIELDS {header_fields}])")
        if status != "OK":
            continue
        for part in msg_data:
            if isinstance(part, tuple):
                msg = email.message_from_bytes(part[1])
                from_addr = getaddresses([msg.get("From", "")])
                to_addrs = getaddresses([msg.get("To", "")])
                cc_addrs = getaddresses([msg.get("Cc", "")])
                date_raw = msg.get("Date", "")
                try:
                    date_parsed = parsedate_to_datetime(date_raw) if date_raw else None
                except Exception:
                    date_parsed = None
                records.append({
                    "folder": folder_name,
                    "from_name": decode_str(from_addr[0][0]) if from_addr else "",
                    "from_email": (from_addr[0][1] if from_addr else "").lower(),
                    "to_emails": [a[1].lower() for a in to_addrs],
                    "cc_emails": [a[1].lower() for a in cc_addrs],
                    "subject": decode_str(msg.get("Subject", "")),
                    "date": date_parsed,
                    "list_unsubscribe": msg.get("List-Unsubscribe", ""),
                })
        print(f"  ... {min(i + batch_size, len(ids))}/{len(ids)}")

    return records


## Step 3 — Fetch Inbox and Sent

In [7]:
inbox_records = fetch_headers(imap, INBOX_FOLDER_NAME, MAX_MESSAGES)


INBOX: 110 messages to scan
  ... 110/110


In [8]:
sent_records = fetch_headers(imap, SENT_FOLDER_NAME, MAX_MESSAGES)


Sent: 500 messages to scan
  ... 200/500
  ... 400/500
  ... 500/500


In [9]:
imap.logout()
print("Done fetching, connection closed.")


Done fetching, connection closed.


## Step 4 — Detect subscriptions

A message is treated as a **subscription/service** if either:
- It carries a `List-Unsubscribe` header (the standard signal used by newsletters, retailers, SaaS tools, etc.), or
- The sender address matches common automated patterns (`noreply@`, `no-reply@`, `newsletter@`, `notifications@`, `updates@`, `info@`, `donotreply@`, `mailer@`, `support@`)

Results are grouped by sender so you get one row per service, with a message count and the most recent subject as a hint.

In [10]:
AUTOMATED_PATTERNS = re.compile(
    r"(no.?reply|newsletter|notification|donotreply|mailer|updates?@|info@|support@|alerts?@|marketing@)",
    re.IGNORECASE,
)

def is_subscription(record):
    if record.get("list_unsubscribe"):
        return True
    if AUTOMATED_PATTERNS.search(record.get("from_email", "")):
        return True
    return False

subscription_groups = defaultdict(lambda: {"count": 0, "name": "", "last_subject": "", "last_date": None, "has_unsubscribe": False})

for r in inbox_records:
    if not is_subscription(r):
        continue
    key = r["from_email"]
    g = subscription_groups[key]
    g["count"] += 1
    g["name"] = r["from_name"] or g["name"]
    if r.get("list_unsubscribe"):
        g["has_unsubscribe"] = True
    if r["date"] and (g["last_date"] is None or r["date"] > g["last_date"]):
        g["last_date"] = r["date"]
        g["last_subject"] = r["subject"]

subs_df = pd.DataFrame([
    {
        "sender_email": email_addr,
        "sender_name": g["name"],
        "message_count": g["count"],
        "has_list_unsubscribe": g["has_unsubscribe"],
        "last_subject": g["last_subject"],
        "last_date": g["last_date"],
    }
    for email_addr, g in subscription_groups.items()
]).sort_values("message_count", ascending=False)

subs_df


,sender_email,sender_name,message_count,has_list_unsubscribe,last_subject,last_date
5,nytdirect@nytimes.com,The New York Times,11,True,"The Morning: Buy now, pay later",2026-08-18 10:51:45+00:00
18,info@mail.tijd.be,De Tijd,7,True,Focus op de essentie: Lees 1 maand voor maar € 1,2026-08-18 03:07:10-06:00
12,breakingnews@nytimes.com,The New York Times,7,True,Breaking news: Border construction in Big Bend...,2026-08-17 19:50:22+00:00
22,jobalerts-noreply@linkedin.com,LinkedIn Job Alerts,6,True,Principal Statistical Methodologist at UCB,2026-08-18 09:40:58+00:00
6,editorpicks@nytimes.com,The New York Times,6,True,Opinion: This is the formula for Democrats in ...,2026-08-17 21:14:47+00:00
8,sales@themusiczoo.com,The Music Zoo,4,True,Shop Our Favorite Acoustics In Stock Now!,2026-08-17 17:03:15+00:00
28,middagupdate@mail.standaard.be,De Standaard Middagupdate,2,True,"Eén klimaatramp, zeven klimaatministers, en ve...",2026-08-18 09:55:36+00:00
9,no-reply@zimmo.be,Zimmo,2,True,Nieuwe panden op Zimmo,2026-08-16 03:49:22+00:00
19,ochtendupdate@mail.standaard.be,De Standaard Ochtendupdate,2,True,Brand in de Hoge Venen blijft opflakkeren: “De...,2026-08-18 06:05:48+00:00
20,noreply@burgerprofiel.vlaanderen.be,Vlaamse overheid,2,False,Aanslagbiljet onroerende voorheffing - aanslag...,2026-08-17 22:50:10+00:00


## Step 5 — Build the full contact list

Everyone you should consider notifying about your new address: anyone who emailed you, and anyone you emailed (To/Cc).

In [ ]:
contact_info = defaultdict(lambda: {"name": "", "received_from": 0, "sent_to": 0, "last_contact": None})

for r in inbox_records:
    if not r["from_email"]:
        continue
    c = contact_info[r["from_email"]]
    c["name"] = r["from_name"] or c["name"]
    c["received_from"] += 1
    if r["date"] and (c["last_contact"] is None or r["date"] > c["last_contact"]):
        c["last_contact"] = r["date"]

for r in sent_records:
    for addr in r["to_emails"] + r["cc_emails"]:
        if not addr:
            continue
        c = contact_info[addr]
        c["sent_to"] += 1
        if r["date"] and (c["last_contact"] is None or r["date"] > c["last_contact"]):
            c["last_contact"] = r["date"]

contacts_df = pd.DataFrame([
    {
        "email": addr,
        "name": info["name"],
        "emails_received_from_them": info["received_from"],
        "emails_you_sent_them": info["sent_to"],
        "last_contact": info["last_contact"],
    }
    for addr, info in contact_info.items()
    if addr != EMAIL_ADDRESS.lower()
]).sort_values("last_contact", ascending=False)

contacts_df


## Step 6 — Export to Excel

In [ ]:
subs_df.to_excel("subscriptions.xlsx", index=False)
contacts_df.to_excel("contacts.xlsx", index=False)
print("Saved subscriptions.xlsx and contacts.xlsx in the current folder.")


## Notes / tuning tips

- **Missed subscriptions**: some services don't set `List-Unsubscribe` and use a normal-looking sender address. Sort `contacts.xlsx` by `emails_received_from_them` — high-frequency senders you don't recognize personally are usually services too.
- **Rerun with a higher `MAX_MESSAGES`** (or `None`) once the first pass looks right, to cover your whole mailbox history.
- **Old address in the From/To of your own sent mail**: if you've already started using a new address, filter `sent_records` further to be sure you're not emailing yourself.
- **Rate limiting**: if Telenet's IMAP server throttles or disconnects on very large mailboxes, reduce `batch_size` in `fetch_headers` (e.g. to 50) and rerun.
